# A1.1 · The reference architecture for agentic AI

**Function A — Securing AI Architectures → TripBot's Architecture, and Every Risk It Carries**  ·  *Both directions*

Builds on **[A1.0 · Start here — what securing an AI architecture means](https://spbreed.github.io/cyber-commons/lessons/A1.0.html)**.

| | |
|---|---|
| Open-source tooling | kagent, OpenTelemetry |
| Open-weight models | Llama 3.3, GLM-4.6 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

"Secure the agent" is not an instruction. It becomes one the moment you can point at a component and a boundary — and every risk in this chapter, every control in the next two, and every detection in Function D names something on the picture you are about to draw.

## 2 · The framework

```
                      +-----------+
   request ---------->|  ingress  |
                      +-----+-----+
                            v
                   +----------------+        +-------------+
                   |  orchestrator  |------->|  messaging  |---> peers
                   +--------+-------+        +-------------+
                            v
                  +-------------------+      +-------------+
                  |   agent runtime   |<---->|    model    |
                  | plan . act . obs  |      |  (predicts) |
                  +----+---------+----+      +-------------+
                       |         |
            +----------v-+   +---v---------------+
            | tools /MCP |   | knowledge /memory |
            +------+-----+   +-------------------+
                   v
              +---------+
              | egress  |
              +---------+

   identity + policy wrap every arrow · observability records every arrow
   trust 0 (an outsider can write here): mcp, knowledge, and the corpus
```

These components exist under different product names in every agent platform,
and the risks attach to the component rather than to the brand.

**Ingress.** Where a request enters: a chat surface, an API call, a webhook, a
scheduled trigger, another system. It carries the requester's identity and
whatever text they supplied.

**Orchestrator.** Decides which agent handles what, and in what pattern. In a
single-agent system this is a few lines; in a multi-agent one it holds the whole
design.

**Agent runtime.** The loop — plan, call a tool, observe the result, decide
again, stop. This is the component that turns text into consequence.

**Model.** Predicts tokens. Holds no credential, opens no socket, changes
nothing. Most of what people fear "the model doing" is done by the runtime.

**Tools and MCP servers.** The only components that change anything. An MCP
server is a third party's process whose tool descriptions land in your context.

**Knowledge and memory.** Retrieval pulls documents in at query time; memory
persists state across turns and sessions. Both inject text the user did not
write.

**Messaging.** The agent-to-agent channel in a multi-agent pattern.

**Identity and policy.** Who is calling, on whose behalf, and whether this call
is permitted. These wrap every other component.

**Egress.** Where data is allowed to go — the last boundary before it leaves.

**Observability.** What can be reconstructed afterwards.

Three of them carry content an outsider can author: **MCP**, **knowledge** and
the corpus behind it. Those are the input surface for the next fifteen lessons.

## 3 · Pattern 1 — the single agent

One loop, one set of tools. Almost everything in production today.

```
  user --> ingress --> agent runtime --> tools --> egress
                          |     ^
                          v     |
                        model (predicts; changes nothing)

  identity + policy wrap every arrow · observability records every arrow
```

The edge that matters is `agent runtime -> tools`. That is where text becomes
consequence, and it exists in every pattern below.

## 4 · Pattern 2 — orchestrator and workers

One planner fans work out to specialised workers and joins the results. The
pattern most multi-agent platforms mean when they say "multi-agent".

```
                            +--> worker A --> tools
  user --> orchestrator ----+--> worker B --> tools
                            +--> worker C --> tools
                                   |
                            join / summarise
```

New surface: the orchestrator decides *who* runs, so anything that can influence
its routing decides which permissions get used.

## 5 · Pattern 3 — sequential handoff

A pipeline of agents, each taking the previous one's output as its input.

```
  user --> agent 1 --> agent 2 --> agent 3 --> result
            recon      analyse     report

  each hop inherits the previous hop's claims; nobody re-checks them
```

New surface: an unverified claim at hop 1 is a fact by hop 3. This is the shape
that makes cascading hallucination (A1.11) a systems problem rather than a model
problem.

## 6 · Pattern 4 — peer swarm over shared memory

Agents with no central planner, coordinating through state they all write to.

```
     +--> agent A --+
  user +--> agent B --+--> shared memory <--+
     +--> agent C --+          ^            |
                               |            |
                    every agent reads what any agent wrote
```

New surface: memory is a write-once, read-forever channel between agents. One
poisoned entry is read back as trusted context indefinitely (A1.4), and there is
no orchestrator to notice.

## 7 · Pattern 5 — a workflow with agent steps

Deterministic control flow, with one or two steps handed to an agent. The
pattern with the best safety properties, and the one people skip.

```
  [fetch] --> [validate] --> (( agent step )) --> [approve] --> [commit]
      deterministic          non-deterministic        deterministic

  the blast radius of the agent is bounded by the two steps either side
```

New surface: almost none, which is the point. If the work fits this shape, the
other four patterns are a cost you do not have to pay.

## 8 · The two components that appear inside all five

Retrieval and human approval are not patterns of their own — they attach to any
of the five above, and each brings one surface with it.

```
  retrieval          agent --> retriever --> corpus
                       ^                       |
                       +----- documents -------+
                       anyone who can write to the corpus writes to the context

  human approval     agent --> proposed action --> [ human ] --> tools
                                                       ^
                       requests arrive faster than a person can read them
```

Retrieval is how outside text reaches the context window without anyone typing
it (A1.3). Approval is a real control for rare irreversible actions and a rubber
stamp for everything else (A1.14).

## 9 · The edge every pattern shares

```
                    +---------------+       +-------+
   untrusted text   | agent runtime | ----> | tools |   consequence
   ---------------> |               |       +-------+
                    +---------------+
                       ^        ^
                  knowledge   messaging
                    memory      MCP
```

Whatever the topology, something reaches the agent runtime and the agent runtime
reaches tools. Every risk in the rest of this chapter is a route into the left
side of that picture. Every control in chapters 2 and 3 is an attempt to stand
somewhere on the arrow.

## What this gives you

You can draw one agentic system you run as thirteen named components, say which of the five patterns it is, and name the three components in it whose content an outsider can author. That list is the input surface for the fifteen risk lessons that follow.

## Your turn

Draw your own system on one page, then mark the `agent_runtime -> tools` edge on it. Everything in chapters 2 and 3 is an argument about what is allowed to stand on that arrow, and you will get more out of them having drawn it first.

---

**Next → [A1.2 · Prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*